In [ ]:
import uproot
import math
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import os
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import sys
from tqdm import tqdm

In [ ]:
files =uproot.open(f"/Users/danielcarber/Documents/SBND/Noise_Analysis/data/fft_output_run14784.root")
files1 =uproot.open(f"/Users/danielcarber/Documents/SBND/Noise_Analysis/data/noise_output_full_14784.root")
print(files1['tpc_noise;5'].keys())
print(files.keys()[0])

In [ ]:
wire_plane_list = ['UB','VB','YB','UA','VA','YA']
wire_df = {'Channel_id':[],'cryo':[],'tpc':[],'tpc':[],'plane':[],'rel_wire':[],'x_0':[],'y_0':[],'z_0':[],'x_1':[],'y_1':[],'z_1':[],'r':[]}
wire_txt = '/Users/danielcarber/Documents/SBND/Noise_Analysis/sbn_noise_repo/sbnd/datafiles/Wire_lengths.txt'

with open(wire_txt) as f:
    for line in f:
        #print(line)
        currentline = line.split(" ")
        #print(currentline)
        wire_df['Channel_id'].append(int(currentline[0]))
        wire_df['cryo'].append(int(currentline[1]))
        wire_df['tpc'].append(int(currentline[2]))
        wire_df['plane'].append(int(currentline[3]))
        wire_df['rel_wire'].append(int(currentline[4]))
        wire_df['x_0'].append(float(currentline[5]))
        wire_df['y_0'].append(float(currentline[6]))
        wire_df['z_0'].append(float(currentline[7]))
        wire_df['x_1'].append(float(currentline[8]))
        wire_df['y_1'].append(float(currentline[9]))
        z_1 = currentline[10]
        #print(z_1[:-2])
        wire_df['z_1'].append(float(currentline[10][:-2]))
        length = np.sqrt(np.square(float(currentline[8])-float(currentline[5]))+np.square(float(currentline[9])-float(currentline[6]))+np.square(float(currentline[10][:-2])-float(currentline[7])))
        wire_df['r'].append(length)
wire_df = pd.DataFrame(wire_df) 
print(wire_df)

In [ ]:
#Puts ffts data into an array

raw_rms_int = files1['tpc_noise;5']['avg_FFT'].array()
raw_rms_coh = files1['tpc_noise;5']['coh_FFT'].array()
print(raw_rms_int[1709*1:1709*2])
print(len(raw_rms_int)/1709)
FFTs_list_int = {}
FFTs_list_coh = {}


In [ ]:
#Breaks the ffts array into its corresponding channel and makes an FFT of the entire detector
channel=-1
for i in tqdm(range(11264)):
    
    if raw_rms_int[1709*(i+1)-1]==0:
        FFTs_list_int[f'{i}'] = 'skip'
        #print(i)
        continue
    FFTs_list_int[f'{i}'] =list((raw_rms_int[i*1709:(i+1)*1709]/raw_rms_int[1709*(i+1)-1])*(1800/4095))
    print(raw_rms_int[1709*(i+1)-1])
#print(FFTs_list_int['0'][-1])


In [ ]:
channel=-1
for i in tqdm(range(11264)):
    
    if raw_rms_coh[1709*(i+1)-1]==0:
        FFTs_list_coh[f'{i}'] = 'skip'
        continue
    FFTs_list_coh[f'{i}'] =list((raw_rms_coh[i*1709:(i+1)*1709]/raw_rms_coh[1709*(i+1)-1])*(1800/4095))
print(FFTs_list_coh['0'][-1])


In [ ]:
C_int={}
fft_bin_int = {}
for channel in FFTs_list_int.keys():
    if FFTs_list_int[channel] =='skip':
        fft_bin_int[channel] = None
        C_int[channel]= None
        continue
    fft = FFTs_list_int[channel]
    C_int[channel] = np.median(fft[1500:1708])
    fft_wl =  np.sqrt(np.power(fft,2) - np.power(C_int[channel],2))
    #print(fft_wl[100])
    fft_wl = [0 if np.isnan(x) else x for x in fft_wl]
    wl = wire_df['r'][wire_df['Channel_id'] == int(channel)].values[0]
    #print(wl)
    fft_bin_int[channel] = fft_wl

C_coh ={}
fft_bin_coh = {}
for channel in FFTs_list_coh.keys():
    if FFTs_list_coh[channel] =='skip':
        fft_bin_coh[channel] = None
        C_coh[channel]= None
        continue
    fft = FFTs_list_coh[channel]
    C_coh[channel] = np.median(fft[1500:1708])
    fft_wl =  np.sqrt(np.power(fft,2) - np.power(C_coh[channel],2))
    fft_wl = [0 if np.isnan(x) else x for x in fft_wl]
    wl = wire_df['r'][wire_df['Channel_id'] == int(channel)].values[0]
    fft_bin_coh[channel] = fft_wl
#print(fft_bin_int['0'])

In [ ]:
#Combine bins to smooth out spectra
fft_smooth_bin_int = {}
step_size = 10
for ch in tqdm(fft_bin_int.keys()):
    if fft_bin_int[ch] is None:
        fft_smooth_bin_int[ch] = None
        continue
    fft_smooth_bin_int[ch] = []
    for i in range(0,1000000,step_size):
        freq_bin = i * 10**-6
        avg_bin = np.mean(fft_bin_int[ch][i:i+step_size])
        fft_smooth_bin_int[ch].append(avg_bin)

In [ ]:
freq = list(range(len(FFTs_list_int['100'])-1))
freq = (np.add(freq,.5))*2/3415
color = ['red','green','blue']
run_number='14784'
C = np.median(FFTs_list_int['0'][1500:1708])
print(C)
fig = make_subplots(rows=1,cols =1,subplot_titles = ('FFT Spectrum Components for Channel 3094',))
fig.add_trace(go.Scatter(x=freq,y =FFTs_list_coh['3094'][1:1708],marker_color = color[0],opacity = 2/(1+1),name = f'Run {run_number} Coherent'),row = 1, col = 1)
fig.add_trace(go.Scatter(x=freq,y = FFTs_list_int['3094'][1:1708],marker_color = color[1],opacity = 2/(1+1),name = f'Run {run_number} Intrinsic'),row = 1, col = 1)

fig.update_xaxes(title_text = "Frequency [MHz]",row = 1, col = 1)
fig.update_yaxes(title_text = "Magnitude [mV/0.59kHz]",row = 1, col = 1)
fig.update_layout(xaxis = dict(tickmode = 'linear',dtick = .1,range= [0,1]))
fig.update_layout(height = 600, width = 900,showlegend = True)
fig.update_annotations(font_size=36)
fig
fig.update_layout(font = dict(size=20),legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.64,
    bgcolor="LightSteelBlue",
        bordercolor="Black",
))

fig.add_annotation(dict(font = dict(size = 25,color="Black",)),xshift= 250,yshift=100,text = f"SBND<br>Preliminary Data",showarrow = False)
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise_Analysis/Plots/SBND_TPC_FFT.png')
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise_Analysis/Plots/SBND_TPC_FFT.pdf')
fig.show()

In [ ]:
freq = list(range(len(FFTs_list_int['100'])-1))
freq = (np.add(freq,.5))*2/3415
freq_smooth = list(range(len(fft_smooth_bin_int['100'])-1))
freq_smooth = (np.add(freq_smooth,.5))*2/3415*step_size
color = ['red','green','blue']
run_number='14784'
C = np.median(FFTs_list_int['0'][1500:1708])
print(C)
fig = make_subplots(rows=1,cols =1,subplot_titles = ('Intrinsic FFT Spectrum',))
fig.add_trace(go.Scatter(x=freq,y =fft_bin_int['1983'][1:1708],marker_color = color[0],opacity = 2/(1+1),name = f'Run {run_number} Intrinsic'),row = 1, col = 1)
#fig.add_trace(go.Scatter(x=freq,y = fft_bin_coh['1983'][1:1708],marker_color = color[1],opacity = 2/(1+1),name = f'Run {run_number} Coherent'),row = 1, col = 1)
fig.add_trace(go.Scatter(x=freq_smooth,y =fft_smooth_bin_int['1983'][1:1708],marker_color = color[2],opacity = 2/(1+1),name = f'Run {run_number} Intrinsic Smoothed'),row = 1, col = 1)

fig.update_xaxes(title_text = "Frequency [MHz]",row = 1, col = 1)
fig.update_yaxes(title_text = "Magnitude [mV/0.59kHz]",row = 1, col = 1)
fig.update_layout(xaxis = dict(tickmode = 'linear',dtick = .1,range= [0,1]))
fig.update_layout(height = 600, width = 900,showlegend = True)
fig.update_annotations(font_size=36)
fig
fig.update_layout(font = dict(size=20),legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.5,
    bgcolor="LightSteelBlue",
        bordercolor="Black",
))

#fig.add_annotation(dict(font = dict(size = 25,color="Black",)),xshift= 250,yshift=100,text = f"SBND<br>Preliminary Data",showarrow = False)
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise_Analysis/Plots/SBND_TPC_FFT.png')
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise_Analysis/Plots/SBND_TPC_FFT.pdf')
fig.show()

In [ ]:
#Format FFTs to Noise spectra input
with open('data_int.txt', 'w') as f:
    lst = [freq,fft_bin_int['1983'][1:1708], fft_bin_int['1100'][1:1708], fft_bin_int['3967'][1:1708],fft_bin_int['3094'][1:1708],fft_bin_int['5627'][1:1708]]
    f.write("2.0 MHz 3415 Ticks 14.0 mV/fC 2.2 microsecond\n")
    f.write(f"Freq  {wire_df['r'][wire_df['Channel_id'] == 1983].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 1100].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3967].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3094].values[0]:.2f} 399.00 401.00\n")
    f.write("Plane 0 0 1 1 2 2\n")
    f.write(f"-1 {C_int['1983']:.2f} {C_int['1100']:.2f} {C_int['3967']:.2f} {C_int['3094']:.2f} {C_int['5627']:.2f} {C_int['5627']:.2f}\n")
    for x in zip(*lst):
        f.write("{0} {1} {2} {3} {4} {5} {5}\n".format(*x))

In [ ]:
#Format FFTs to Noise spectra input
with open('data_int_smooth.txt', 'w') as f:
    lst = [freq_smooth,fft_smooth_bin_int['1983'][1:1708], fft_smooth_bin_int['1100'][1:1708], fft_smooth_bin_int['1579'][1:1708], fft_smooth_bin_int['3550'][1:1708], 
           fft_smooth_bin_int['3967'][1:1708],fft_smooth_bin_int['3094'][1:1708], fft_smooth_bin_int['2365'][1:1708], fft_smooth_bin_int['3516'][1:1708],
           fft_smooth_bin_int['3970'][1:1708],fft_smooth_bin_int['11260'][1:1708]]
    f.write("2.0 MHz 3415 Ticks 14.0 mV/fC 2.2 microsecond\n")
    f.write(f"Freq  {wire_df['r'][wire_df['Channel_id'] == 1983].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 1100].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 1579].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3550].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3967].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3094].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 2365].values[0]:.2f} {wire_df['r'][wire_df['Channel_id'] == 3516].values[0]:.2f} 399.00 401.00\n")
    f.write("Plane 0 0 0 0 1 1 1 1 2 2\n")
    f.write(f"-1 {C_int['1983']:.2f} {C_int['1100']:.2f} {C_int['2395']:.2f} {C_int['1579']:.2f} {C_int['3967']:.2f} {C_int['3094']:.2f} {C_int['2365']:.2f} {C_int['3516']:.2f} {C_int['1133']:.2f} {C_int['2339']:.2f}\n")
    for x in zip(*lst):
        f.write("{0} {1} {2} {3} {4} {5} {6} {7} {8} {9} {10}\n".format(*x))